In [ ]:
!pip install -q torch scikit-learn numpy matplotlib

import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from google.colab import drive

drive.mount('/content/drive')

SAVE_PATH = '/content/drive/MyDrive/ptb-xl-dataset/'
CKPT_PATH = '/content/drive/MyDrive/ptb-xl-dataset/'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Load data
X_train_norm = np.load(SAVE_PATH + 'X_train_norm.npy')
X_test_norm  = np.load(SAVE_PATH + 'X_test_norm.npy')
y_train_enc  = np.load(SAVE_PATH + 'y_train_enc.npy')
y_test_enc   = np.load(SAVE_PATH + 'y_test_enc.npy')

# Fold-9 split (same as notebook 05): folds 1-8 = train, fold 9 = val
Y = pd.read_csv(SAVE_PATH + 'ptb-xl/ptbxl_database.csv', index_col='ecg_id')
train_meta = Y[Y.strat_fold != 10]
train_mask = (train_meta.strat_fold != 9).values
X_train, y_train = X_train_norm[train_mask], y_train_enc[train_mask]

print(f"Train: {X_train.shape} | Test: {X_test_norm.shape} | Device: {device}")


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=7, stride=stride, padding=3, bias=False)
        self.bn1   = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=7, padding=3, bias=False)
        self.bn2   = nn.BatchNorm1d(out_channels)
        self.skip  = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.skip(x)
        return F.relu(out)


class ECGEncoder(nn.Module):
    def __init__(self, embedding_dim=256):
        super().__init__()
        self.conv1   = nn.Conv1d(12, 64, kernel_size=15, stride=2, padding=7, bias=False)
        self.bn1     = nn.BatchNorm1d(64)
        self.layer1  = ResidualBlock(64,  64)
        self.layer2  = ResidualBlock(64,  128, stride=2)
        self.layer3  = ResidualBlock(128, 256, stride=2)
        self.layer4  = ResidualBlock(256, 512, stride=2)
        self.pool    = nn.AdaptiveAvgPool1d(1)
        self.project = nn.Linear(512, embedding_dim)
    def forward(self, x):
        x = x.transpose(1, 2)
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        x = self.pool(x).squeeze(-1)
        return self.project(x)


class FTDataset(Dataset):
    def __init__(self, signals, labels):
        self.signals = torch.FloatTensor(signals)
        self.labels  = torch.FloatTensor(labels)
    def __len__(self): return len(self.signals)
    def __getitem__(self, idx): return self.signals[idx], self.labels[idx]

print("Setup done.")

Mounted at /content/drive
Train: (17418, 1000, 12) | Test: (2198, 1000, 12) | Device: cuda
Setup done.


In [ ]:
SEEDS = [1, 2, 3, 4, 5]

def set_seeds(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def extract_embeddings(encoder, data, bs=256):
    encoder.eval()
    loader = DataLoader(TensorDataset(torch.FloatTensor(data)), batch_size=bs, shuffle=False)
    out = []
    with torch.no_grad():
        for (b,) in loader:
            out.append(encoder(b.to(device)).cpu().numpy())
    return np.concatenate(out, axis=0)

def load_encoder(ckpt_name):
    enc = ECGEncoder(256).to(device)
    enc.load_state_dict(torch.load(CKPT_PATH + ckpt_name, map_location=device)['encoder_state'])
    return enc

# Pre-extract frozen embeddings once (encoders don't change across seeds)
simclr_enc = load_encoder('simclr_checkpoint_epoch50.pt')
cpc_enc    = load_encoder('cpc_checkpoint_epoch50.pt')
simclr_tr_emb = extract_embeddings(simclr_enc, X_train); simclr_te_emb = extract_embeddings(simclr_enc, X_test_norm)
cpc_tr_emb    = extract_embeddings(cpc_enc, X_train);    cpc_te_emb    = extract_embeddings(cpc_enc, X_test_norm)

def probe(train_emb, test_emb, y_tr_all, idx):
    clf = OneVsRestClassifier(LogisticRegression(max_iter=1000))
    clf.fit(train_emb[idx], y_tr_all[idx])
    prob = clf.predict_proba(test_emb); pred = clf.predict(test_emb)
    return roc_auc_score(y_test_enc, prob, average='macro'), f1_score(y_test_enc, pred, average='macro')

def probe_raw(idx):
    clf = OneVsRestClassifier(LogisticRegression(max_iter=1000))
    clf.fit(X_train[idx].reshape(len(idx), -1), y_train[idx])
    Xte = X_test_norm.reshape(len(X_test_norm), -1)
    prob = clf.predict_proba(Xte); pred = clf.predict(Xte)
    return roc_auc_score(y_test_enc, prob, average='macro'), f1_score(y_test_enc, pred, average='macro')

def finetune(ckpt_name, idx, seed, pretrained=True):
    set_seeds(seed)
    enc = load_encoder(ckpt_name) if pretrained else ECGEncoder(256).to(device)
    model = nn.Sequential(enc, nn.Linear(256, 5), nn.Sigmoid()).to(device)
    g = torch.Generator(); g.manual_seed(seed)
    loader = DataLoader(FTDataset(X_train[idx], y_train[idx]), batch_size=64, shuffle=True, generator=g)
    opt = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    crit = nn.BCELoss()
    model.train()
    for _ in range(20):
        for sig, lab in loader:
            sig, lab = sig.to(device), lab.to(device)
            loss = crit(model(sig), lab)
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    loader = DataLoader(FTDataset(X_test_norm, y_test_enc), batch_size=64, shuffle=False)
    probs = []
    with torch.no_grad():
        for sig, _ in loader:
            probs.append(model(sig.to(device)).cpu().numpy())
    probs = np.concatenate(probs)
    preds = (probs > 0.5).astype(int)
    return roc_auc_score(y_test_enc, probs, average='macro'), f1_score(y_test_enc, preds, average='macro')

# Collect results
lp = {k: {f: [] for f in [0.05, 0.10]} for k in ['Raw', 'SimCLR', 'CPC']}
ft = {k: {f: [] for f in [0.05, 0.10]} for k in ['Scratch', 'SimCLR', 'CPC']}

for seed in SEEDS:
    print(f"===== Seed {seed} =====")
    for frac in [0.05, 0.10]:
        set_seeds(seed)
        n = int(len(X_train) * frac)
        idx = np.random.choice(len(X_train), n, replace=False)

        # Linear probing
        lp['Raw'][frac].append(probe_raw(idx))
        lp['SimCLR'][frac].append(probe(simclr_tr_emb, simclr_te_emb, y_train, idx))
        lp['CPC'][frac].append(probe(cpc_tr_emb, cpc_te_emb, y_train, idx))

        # Fine-tuning
        ft['Scratch'][frac].append(finetune(None, idx, seed, pretrained=False))
        ft['SimCLR'][frac].append(finetune('simclr_checkpoint_epoch50.pt', idx, seed))
        ft['CPC'][frac].append(finetune('cpc_checkpoint_epoch50.pt', idx, seed))

def show(title, table, keys):
    print(f"\n===== {title} (mean ± std over 5 seeds) =====")
    for k in keys:
        for frac in [0.05, 0.10]:
            arr = np.array(table[k][frac])
            m, s = arr.mean(axis=0), arr.std(axis=0)
            print(f"{k:8s} {int(frac*100):2d}% | AUC: {m[0]:.4f}±{s[0]:.4f} | F1: {m[1]:.4f}±{s[1]:.4f}")

show("LINEAR PROBING", lp, ['Raw', 'SimCLR', 'CPC'])
show("FINE-TUNING", ft, ['Scratch', 'SimCLR', 'CPC'])

===== Seed 1 =====
===== Seed 2 =====
===== Seed 3 =====
===== Seed 4 =====
===== Seed 5 =====

===== LINEAR PROBING (mean ± std over 5 seeds) =====
Raw       5% | AUC: 0.4977±0.0050 | F1: 0.2537±0.0085
Raw      10% | AUC: 0.4996±0.0033 | F1: 0.2732±0.0057
SimCLR    5% | AUC: 0.8017±0.0074 | F1: 0.5248±0.0155
SimCLR   10% | AUC: 0.8135±0.0044 | F1: 0.5343±0.0118
CPC       5% | AUC: 0.8234±0.0043 | F1: 0.5570±0.0096
CPC      10% | AUC: 0.8341±0.0025 | F1: 0.5722±0.0038

===== FINE-TUNING (mean ± std over 5 seeds) =====
Scratch   5% | AUC: 0.8587±0.0026 | F1: 0.5847±0.0338
Scratch  10% | AUC: 0.8693±0.0042 | F1: 0.6104±0.0305
SimCLR    5% | AUC: 0.8406±0.0085 | F1: 0.5998±0.0177
SimCLR   10% | AUC: 0.8572±0.0124 | F1: 0.6087±0.0288
CPC       5% | AUC: 0.8572±0.0041 | F1: 0.6268±0.0046
CPC      10% | AUC: 0.8647±0.0108 | F1: 0.6297±0.0268
